# Parda — web app M3, step A: YOLO golden cases for the browser port

Runs the **shipped** YOLO (parda-yolo.onnx, 10 MB) on 8 real benchmark pages with faces, signatures, QR codes and
stamps, and records every step ultralytics takes: the 1024x1024 network input, the raw network output and the final
boxes. The browser version must reproduce them. Also saves ultralytics' own code for these steps.

**Settings:** Accelerator **None (CPU)** · Internet **On**
**Inputs:** Your Work → **Parda** (Phase 1) · Datasets tab → `celeba-dataset` · **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`
Run all cells (~10 min), then download **`yolo_goldens.zip`** from the Output panel and attach it in the chat.


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. ultralytics + ONNX runtime
!pip install -q ultralytics onnxruntime
!python -c "import ultralytics, onnxruntime; print('ultralytics', ultralytics.__version__, '| onnxruntime', onnxruntime.__version__)"


In [ ]:
# 3. The benchmark pages (real faces, as in Phase 4) + the shipped YOLO from your HF repo
import json, shutil
from huggingface_hub import HfApi, hf_hub_download
from parda.ocr.run_ocr import find_data
from parda.vision.faces import find_faces_dir
W = "/kaggle/working"
DATA = find_data()
FACES = find_faces_dir("/kaggle/input", exclude=[DATA])
assert DATA and FACES, "attach the Parda (Phase 1) notebook and the celeba-dataset"
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
YOLO = hf_hub_download(f"{HF_USER}/parda-onnx-v1", "yolo/parda-yolo.onnx", token=os.environ["HF_TOKEN"])
print("YOLO:", YOLO, f"{os.path.getsize(YOLO) / 2**20:.1f} MB")
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
!ls {W}/yolo_bench/images/bench | wc -l


In [ ]:
# 4. Record (~3 min)
!python tools/make_yolo_goldens.py --yolo {YOLO} --images {W}/yolo_bench/images/bench --labels {W}/yolo_bench/labels/bench --n 8 --out {W}/yolo_goldens 2>&1 | grep -v -i warning | tail -30


In [ ]:
# 5. One zip to download (Output -> yolo_goldens.zip)
shutil.make_archive(f"{W}/yolo_goldens", "zip", f"{W}/yolo_goldens")
print(sorted(os.listdir(f"{W}/yolo_goldens")), f"| yolo_goldens.zip: {os.path.getsize(f'{W}/yolo_goldens.zip') / 2**20:.1f} MB")
